### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="otto_group_product_classification_challenge",
    dataset_year="2015",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/otto-group-product-classification-challenge",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c otto-group-product-classification-challenge -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/otto_group_product_classification_challenge && mv train.csv local-data-warehouse/otto_group_product_classification_challenge/
""",
    # References
    academic_reference_bibtex=r"""@misc{Bossan2015OttoGroupProductClassificationChallenge,
  author = {Benjamin Bossan and Josef Feigl and Wendy Kan},
  title  = {Otto Group Product Classification Challenge},
  year   = {2015},
  howpublished = {\url{https://kaggle.com/competitions/otto-group-product-classification-challenge}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Bossan2015OttoGroupProductClassificationChallenge",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized, so feature meanings are unknown.
- Features and data is clean. Top Kaggle experts applied no relevant preprocessing.
- We drop the index column.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="target",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)
df = df.drop(columns=["id"])
df["target"] = df["target"].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()